# $\mathbb{Z}_2^F \times \mathbb{Z}_2^T$ Hamiltonians - open boundary conditions

Created: 20-07-2026

Build off [this notebook](z2_f_x_z2_t_hamiltonians.ipynb), now sweep and save states.

# Imports

In [1]:
from time import time

In [2]:
import numpy as np

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from itertools import product

In [6]:
import quimb.tensor as qtn
import quimb as qu

In [7]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

In [8]:
from humanize import naturalsize

# Definitions

In [9]:
group_quads = list(product([0,1], repeat=4))

In [10]:
spin_ops_dict = {
    (0,0): [("I", 1), ("y", 1)],
    (1,1): [("I", 1), ("y", -1)],
    (0,1): [("z", 1), ("x", 1j)],
    (1,0): [("z", 1), ("x", -1j)]
}

In [11]:
def get_fermionic_op_string(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    out_string = ''
    out_indices = list()

    if (g_left + g_out) % 2:
        out_string += '+'
        out_indices.append(0)
    if (g_out + g_right) % 2:
        out_string += '+'
        out_indices.append(1)
    if (g_in + g_right) % 2:
        out_string += '-'
        out_indices.append(1)
    if (g_left + g_in) % 2:
        out_string += '-'
        out_indices.append(0)

    return (out_string, out_indices)

In [12]:
def cocycle_phase(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    exponent = (
        (g_in - g_left)*(g_right - g_in)
        - (g_out - g_left)*(g_right - g_out)
    )
    exponent = exponent % 2

    # i.e. (-1)**exponent
    out = -1 if exponent else 1

    return out

In [13]:
def get_group_quad_terms(group_quad, L, nontriv_cocycle=False,
                         strength_scaling=1):
    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength

        ferm_op_string, ferm_indices = get_fermionic_op_string(group_quad)
        op_string = f"{spin_string}|{ferm_op_string}"

        base_index = [0, 1, 2, *ferm_indices]
        all_indices = [
            [x+i for x in base_index]
            for i in range(L-2)
        ]

        phase = cocycle_phase(group_quad) if nontriv_cocycle else 1
        scaling_constant = strength*strength_scaling*phase
        current_term = [
            op_string, [[scaling_constant, *indices] for indices in all_indices]
        ]

        terms.append(current_term)

    return terms

In [14]:
triv_fermion_terms_phases = [
    ('I', 1),
    ('n', -1)
]

In [15]:
def get_trivial_group_quad_terms(group_quad, L, strength_scaling=1):
    #Case where both cocycle and fermionic decoration are trivial.

    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength


        for pair_1, pair_2 in product(triv_fermion_terms_phases, repeat=2):
            fp_op_1, fp_phase_1 = pair_1
            fp_op_2, fp_phase_2 = pair_2
            
            op_string = f"{spin_string}|{fp_op_1}{fp_op_2}"
    
            base_index = [0, 1, 2, 0, 1]
            all_indices = [
                [x+i for x in base_index]
                for i in range(L-2)
            ]
    
            scaling_constant = strength*strength_scaling*fp_phase_1*fp_phase_2
            current_term = [
                op_string, [[scaling_constant, *indices] for indices in all_indices]
            ]
    
            terms.append(current_term)

    return terms

In [16]:
def get_triv_to_n1_non_triv_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = [
        l for group_quad in group_quads
        for l in get_trivial_group_quad_terms(group_quad, L, 1-t)
    ]

    non_triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, False, t)
    ]

    # For dangling fermion site on right edge, force to zero.
    dummy_fermion_term = [
        [f'|{op}', [[strength, L-1],]]
        for op, strength in triv_fermion_terms_phases
    ]

    all_terms = [*triv_terms, *non_triv_terms, *dummy_fermion_term]

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

In [17]:
t=0

In [18]:
def get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L):
    spin_basis = spin_basis_1d(L)
    fermion_basis = spinless_fermion_basis_1d(L)
    basis = tensor_basis(spin_basis, fermion_basis)

    triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, False, 1-t)
    ]

    non_triv_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms(group_quad, L, True, t)
    ]

    # For dangling fermion site on right edge, force to zero.
    dummy_fermion_term = [
        [f'|{op}', [[strength, L-1],]]
        for op, strength in triv_fermion_terms_phases
    ]

    all_terms = [*triv_terms, *non_triv_terms, *dummy_fermion_term]

    h = hamiltonian(
        all_terms,
        [],
        basis=basis,
        dtype=np.complex128,
        check_symm=False,
        check_herm=False
    )

    return h

# Sweep

In [19]:
parameters = np.linspace(0, 1, 21)

## 8 site

In [20]:
L = 8

## Trivial cocycle

In [21]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=4, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_8_site_ed_obc/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                    | 0/21 [00:00<?, ?it/s]/var/folders/r3/xn8xq5c17932m2g1b4r3dss80000gn/T/ipykernel_18365/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|████▍                                                                                       | 1/21 [00:00<00:08,  2.45it/s]/var/folders/r3/xn8xq5c17932m2g1b4r3dss80000gn/T/ipykernel_18365/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|███████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:33<00:00,  1.59s/it]


## Nontrivial cocycle

In [22]:
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=4, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_8_site_ed_obc/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                    | 0/21 [00:00<?, ?it/s]/var/folders/r3/xn8xq5c17932m2g1b4r3dss80000gn/T/ipykernel_20535/4155063260.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|███████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:43<00:00,  2.06s/it]


## 10 site

In [23]:
L = 10

## Trivial cocycle

In [24]:
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=4, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_10_site_ed_obc/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                    | 0/21 [00:00<?, ?it/s]/var/folders/r3/xn8xq5c17932m2g1b4r3dss80000gn/T/ipykernel_18365/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|███████████████████████████████████████████████████████████████████████████████████████████| 21/21 [21:40<00:00, 61.93s/it]


## Nontrivial cocycle

In [24]:
L=10

for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=4, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_10_site_ed_obc/{t_string}.npz', energy=e, psi=psi)

  0%|                                                                                                    | 0/21 [00:00<?, ?it/s]/var/folders/r3/xn8xq5c17932m2g1b4r3dss80000gn/T/ipykernel_20535/4155063260.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
100%|███████████████████████████████████████████████████████████████████████████████████████████| 21/21 [24:28<00:00, 69.93s/it]


## 12 site
This takes a while...

In [26]:
L = 12

## Trivial cocycle

In [19]:
"""
for t in tqdm(parameters):
    h = get_triv_to_n1_non_triv_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_triv_to_nontriv_n1_12_site_ed/{t_string}.npz', energy=e, psi=psi)
"""

  0%|                                                                                                                                                                                                         | 0/21 [00:00<?, ?it/s]/tmp/ipykernel_3403/2504458043.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|█████████▏                                                                                                                                                                                       | 1/21 [00:55<18:20, 55.03s/it]/tmp/ipykernel_3403/2504458043.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
  5%|████████▋                                                                            

KeyboardInterrupt: 

## Nontrivial cocycle

In [ ]:
"""
for t in tqdm(parameters):
    h = get_nontriv_n1_to_nontriv_cocycle_hamiltonian(t, L)
    e, psi = h.eigsh(k=1, which='SA')

    t_string = str(int(100*t))
    np.savez(rf'../data/z2_f_x_z2_t_nontriv_n1_to_nontriv_cocyle_12_site_ed/{t_string}.npz', energy=e, psi=psi)
"""

# Old code
## Check groundstate degeneracies
Conclusion, there is a 4 fold degeneracy, coming from the choice of spin at the boundaries.

In [102]:
L=8

In [119]:
h = get_triv_to_n1_non_triv_hamiltonian(0, L)
results = h.eigsh(k=16, which='SA')

/tmp/ipykernel_38791/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [120]:
from collections import Counter

In [121]:
Counter(np.rint(results[0]))

Counter({np.float64(-5.0): 8, np.float64(-6.0): 4, np.float64(-4.0): 4})

In [122]:
X = results[1]

In [123]:
np.round(X.conj().T @ X, 3)

array([[ 1.   +0.j   , -0.   +0.j   , -0.011+0.014j, -0.   +0.j   ,
         0.   +0.j   ,  0.022-0.012j, -0.   -0.j   , -0.001+0.005j,
         0.   -0.j   ,  0.   -0.j   , -0.   -0.j   ,  0.   -0.j   ,
        -0.   -0.j   ,  0.   -0.j   , -0.   -0.j   , -0.   +0.j   ],
       [-0.   -0.j   ,  1.   +0.j   ,  0.   +0.j   ,  0.002-0.002j,
         0.   -0.j   , -0.   +0.j   , -0.   -0.j   ,  0.   -0.j   ,
         0.   +0.j   ,  0.004+0.004j, -0.008+0.006j,  0.001-0.006j,
        -0.041+0.028j, -0.   -0.j   ,  0.019+0.009j, -0.002-0.007j],
       [-0.011-0.014j,  0.   -0.j   ,  1.   +0.j   ,  0.   +0.j   ,
         0.   +0.j   , -0.009-0.009j,  0.   +0.j   , -0.06 -0.034j,
        -0.   -0.j   , -0.   +0.j   ,  0.   -0.j   , -0.   -0.j   ,
        -0.   +0.j   , -0.   -0.j   ,  0.   -0.j   ,  0.   -0.j   ],
       [-0.   -0.j   ,  0.002+0.002j,  0.   -0.j   ,  1.   +0.j   ,
         0.   +0.j   ,  0.   -0.j   ,  0.   +0.j   , -0.   -0.j   ,
        -0.   +0.j   ,  0.001-0.03j ,  0.   -

In [124]:
h = get_triv_to_n1_non_triv_hamiltonian(0, 6)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


ArpackError: ARPACK error 3: No shifts could be applied during a cycle of the Implicitly restarted Arnoldi iteration. One possibility is to increase the size of NCV relative to NEV. 

In [125]:
Counter(np.rint(results[0]))

Counter({np.float64(-5.0): 8, np.float64(-6.0): 4, np.float64(-4.0): 4})

In [126]:
h = get_triv_to_n1_non_triv_hamiltonian(0, 4)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [127]:
Counter(np.rint(results[0]))

Counter({np.float64(-1.0): 28, np.float64(-2.0): 4})

In [128]:
h = get_triv_to_n1_non_triv_hamiltonian(1, 8)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2972469077.py:24: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [129]:
np.sort(results[0])

array([-6. , -6. , -6. , -6. , -5.5, -5.5, -5.5, -5.5, -5.5, -5.5, -5.5,
       -5.5, -5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. , -5. ,
       -5. , -5. , -5. , -5. , -5. , -5. , -4.5, -4.5, -4.5, -4.5])

In [71]:
Counter(np.round(results[0], 1))

Counter({np.float64(-5.5): 16, np.float64(-6.0): 8, np.float64(-5.0): 8})

In [72]:
h = get_triv_to_n1_non_triv_hamiltonian(1, 7)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [73]:
Counter(np.round(results[0], 1))

Counter({np.float64(-4.5): 14, np.float64(-4.0): 10, np.float64(-5.0): 8})

In [74]:
h = get_triv_to_n1_non_triv_hamiltonian(1, 6)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [75]:
Counter(np.round(results[0], 1))

Counter({np.float64(-3.5): 16, np.float64(-4.0): 8, np.float64(-3.0): 8})

In [76]:
h = get_triv_to_n1_non_triv_hamiltonian(1, 5)
results = h.eigsh(k=32, which='SA')

/tmp/ipykernel_38791/2657243103.py:18: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(


In [77]:
Counter(np.round(results[0], 1))

Counter({np.float64(-2.5): 12, np.float64(-2.0): 12, np.float64(-3.0): 8})